# Fig. 2 — Spatial propagation fields

In [ ]:
# Setup
# Recompute E1146 events from raw SEEG; Fig1 independently uses Y1.
import _setup
import numpy as np
from analysis import from_events as analysis, figure2 as spatial
from plotting import fig2
from plotting.computational import shared_field_null
from workflows.tutorial import raw_inputs, TutorialOutput, save_fields, save_events
from data_processing import raw_to_events as raw
from workflows.common import write_json
config, raw_root = raw_inputs('raw_example.json')
events = raw.join_blocks([raw.process_record(r, config, raw_root)[2] for r in config['records']])
raw_out = TutorialOutput('fig2_raw')
save_events(raw_out, events, {}, config, raw_root)
import json
xyz = analysis.align_coordinates(events, json.loads((raw_root/config['coordinates']).read_text()))
out = TutorialOutput('fig2')

In [ ]:
# Fig. 2A: Projection geometry
# Fit rank = intercept + beta dot xyz; beta/||beta|| points from earlier to later ranks.
# Build the patient-specific plane from the full Timing+Space templates.
# Draw physical millimetres and the same 6-mm display kernel as the paper.
directions = analysis.fit_event_directions_3d(np.where(events['participating'], events['ranks'], np.nan), xyz)
clustered = analysis.spatial_templates(events, xyz, directions)
field_a = analysis.illustrative_grid(events, clustered, xyz, 0)
field_b = analysis.illustrative_grid(events, clustered, xyz, 1, plane=field_a['plane'])
geometry = spatial.display_geometry(events, xyz, field_a)
out.show(fig2.projection(xyz, geometry), 'fig2_A_geometry')

In [ ]:
# Fig. 2B: Direction and spatial information gain
# Train Timing-only and Timing+Space on alternating recording blocks.
# Assign common held-out events using ranks only; their directions determine the score.
# The rose uses one common basis: timing axes dashed, Timing+Space axes solid.
# E1146 supplies one patient point, not the paper's full cohort.
directions, gain = analysis.direction_and_crossfit(events, xyz, n_null=1000, n_train_null=50)
rose = spatial.heldout_rose(events, xyz, directions)
out.show(fig2.direction_scores(rose, gain), 'fig2_B_crossfit')

In [ ]:
# Fig. 2C: Time-resolved HFO propagation
# Reopen raw SEEG for one event per template and compute its Hilbert envelope and local baseline.
# Use the original Fig2 spectral-readout and dynamic-field painters, frame selection,
# 6-mm physical smoothing, per-frame top-three scaling, and fixed gamma=0.5 color normalization.
# The paper's Fig2 display centroid uses the dominant enhancement component; Fig1 ranks retain S^3.
dynamic = spatial.dynamic_events(events, clustered, geometry, config, raw_root)
out.show(fig2.event_fields(dynamic, geometry), 'fig2_C_dynamic_fields')
write_json(out.path/'event_selection.json', dict(events=[e['event_index'] for e in dynamic['events']],
    times_ms=dynamic['times_ms'], frame_selection=dynamic['selection']))

In [ ]:
# Fig. 2D: Static template fields
# Smooth normalized mean ranks with participation weights in the shared physical plane.
# Keep the paper's viridis early-to-late scale and contact overlays.
rank_fields = [spatial.rank_field(geometry, clustered['templates'][k], clustered['support'][k]) for k in range(2)]
out.show(fig2.template_fields(rank_fields, geometry), 'fig2_D_template_fields')

In [ ]:
# Spatial-field comparison
# Rebuild the qualified shared fields and shuffle complete contact identities to obtain a null.
# This demonstrates the per-patient calculation underlying Fig2F; it is not a cohort test.
field_record = analysis.template_fields(events, clustered, xyz, n_boot=200,
    subject_id=config['subject'], subject=config['subject'], dataset=config.get('dataset', 'raw_demo'))
similarity = analysis.shared_field_similarity(field_record, n_perm=1000)
figure = shared_field_null(similarity)
figure.axes[0].set_title('E1146')
out.show(figure, 'fig2_field_comparison')

In [ ]:
# Save the newly computed fields and scores
save_fields(out, gain, clustered, field_record, field_a, field_b, similarity)